In [2]:
# ============================================================
# SetStop-KGQA: WebQSP State Generation and Baselines V2
# Complete single-cell CPU notebook
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import collections
import hashlib
import json
import random
import re
import time

import numpy as np


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("/content/drive/MyDrive/SetStop_KGQA")
DATA_DIR = ROOT / "data/extracted/webqsp"
PROCESSED = ROOT / "data/processed"
OUTPUTS = ROOT / "outputs/preprocessing_v2"

PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

# First run with True. After it succeeds, change to False.
PILOT = False
PILOT_QUESTIONS_PER_SPLIT = 20

MAX_ANSWERS = 50
MAX_HOPS = 3

# One action expands one (source entity, relation) branch.
MAX_ACTIONS_PER_TRAJECTORY = 12
MAX_CANDIDATES_PER_STATE = 16

# Generate multiple trajectories only for training.
TRAIN_TRAJECTORIES = 3
EVAL_TRAJECTORIES = 1

REQUIRED_FILES = [
    "train_simple.json",
    "dev_simple.json",
    "test_simple.json",
    "entities.txt",
    "relations.txt",
]

for filename in REQUIRED_FILES:
    path = DATA_DIR / filename
    assert path.exists(), f"Missing required file: {path}"

MODE = "pilot" if PILOT else "full"

print("=" * 90)
print("SETSTOP-KGQA PREPROCESSING V2")
print("=" * 90)
print("Runtime: CPU")
print("Mode:", MODE.upper())
print("Data:", DATA_DIR)
print("Maximum graph hops:", MAX_HOPS)
print("Action definition: (source entity, relation) branch")
print("=" * 90)


# ============================================================
# 2. GENERAL HELPERS
# ============================================================

def read_vocabulary(path):
    with open(path, encoding="utf-8") as file:
        return [line.rstrip("\n") for line in file]


ENTITY_NAMES = read_vocabulary(DATA_DIR / "entities.txt")
RELATION_NAMES = read_vocabulary(DATA_DIR / "relations.txt")

ENTITY_TO_ID = {
    entity_name: index
    for index, entity_name in enumerate(ENTITY_NAMES)
}

RELATION_TO_ID = {
    relation_name: index
    for index, relation_name in enumerate(RELATION_NAMES)
}


def iter_jsonl(path):
    with open(path, encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue

            try:
                yield json.loads(line)
            except Exception as error:
                raise ValueError(
                    f"Invalid JSON in {path}, line {line_number}"
                ) from error


def entity_index(value):
    if isinstance(value, (int, np.integer)):
        return int(value)

    if isinstance(value, str):
        return ENTITY_TO_ID.get(value)

    if isinstance(value, dict):
        for key in ["kb_id", "id", "entity"]:
            if key in value:
                return entity_index(value[key])

    return None


def relation_index(value):
    if isinstance(value, (int, np.integer)):
        return int(value)

    if isinstance(value, str):
        return RELATION_TO_ID.get(value)

    return None


def get_question(example):
    for key in ["question", "text", "query"]:
        value = example.get(key)
        if isinstance(value, str):
            return value.strip()

    raise KeyError("Question field was not found")


def get_topic_ids(example):
    raw_entities = example.get(
        "entities",
        example.get("topic_entities", [])
    )

    topic_ids = {
        entity_index(value)
        for value in raw_entities
    }

    return {
        value
        for value in topic_ids
        if value is not None
    }


def get_gold_ids(example):
    gold_ids = {
        entity_index(value)
        for value in example.get("answers", [])
    }

    return {
        value
        for value in gold_ids
        if value is not None
    }


def get_graph(example):
    subgraph = example.get("subgraph", {})

    graph_entities = {
        entity_index(value)
        for value in subgraph.get("entities", [])
    }
    graph_entities.discard(None)

    raw_edges = subgraph.get(
        "tuples",
        subgraph.get("triples", [])
    )

    edges = []

    for row in raw_edges:
        if isinstance(row, dict):
            subject = entity_index(
                row.get("subject", row.get("s"))
            )
            relation = relation_index(
                row.get("relation", row.get("r"))
            )
            object_entity = entity_index(
                row.get("object", row.get("o"))
            )
        else:
            if len(row) != 3:
                continue

            subject = entity_index(row[0])
            relation = relation_index(row[1])
            object_entity = entity_index(row[2])

        if (
            subject is None
            or relation is None
            or object_entity is None
        ):
            continue

        subject = int(subject)
        relation = int(relation)
        object_entity = int(object_entity)

        edges.append(
            (subject, relation, object_entity)
        )

        graph_entities.add(subject)
        graph_entities.add(object_entity)

    return graph_entities, edges


def get_question_id(split, example, line_number):
    for key in ["id", "qid", "question_id"]:
        if key in example:
            return str(example[key])

    question_hash = hashlib.sha1(
        get_question(example).encode("utf-8")
    ).hexdigest()[:12]

    return f"{split}_{line_number}_{question_hash}"


def relation_name(relation_id):
    if 0 <= relation_id < len(RELATION_NAMES):
        return RELATION_NAMES[relation_id]

    return f"relation_{relation_id}"


def stable_seed(text):
    digest = hashlib.sha1(
        text.encode("utf-8")
    ).hexdigest()[:8]

    return SEED + int(digest, 16)


# ============================================================
# 3. SCHEMA CHECK
# ============================================================

sample = next(
    iter_jsonl(DATA_DIR / "train_simple.json")
)

sample_nodes, sample_edges = get_graph(sample)
sample_topics = get_topic_ids(sample)
sample_gold = get_gold_ids(sample)

print("\nSchema check")
print("-" * 90)
print("Top-level keys:", sorted(sample.keys()))
print("Question:", get_question(sample))
print("Topic entities:", len(sample_topics))
print("Gold answers:", len(sample_gold))
print("Graph entities:", len(sample_nodes))
print("Graph edges:", len(sample_edges))

assert sample_topics, "No topic entities were detected"
assert sample_edges, "No graph edges were detected"
assert sample_gold, "No gold answers were detected"

print("Schema check passed.")


# ============================================================
# 4. BUILD MANIFEST AND FILTER MULTI-ANSWER QUESTIONS
# ============================================================

manifest_rows = []
selected_examples = {}

expected_full_counts = {
    "train": 1200,
    "dev": 118,
    "test": 736,
}

for split in ["train", "dev", "test"]:
    selected = []

    source_path = DATA_DIR / f"{split}_simple.json"

    for line_number, example in enumerate(
        iter_jsonl(source_path),
        start=1
    ):
        question_id = get_question_id(
            split,
            example,
            line_number
        )

        gold_ids = get_gold_ids(example)
        topic_ids = get_topic_ids(example)
        graph_nodes, graph_edges = get_graph(example)

        usable = (
            2 <= len(gold_ids) <= MAX_ANSWERS
            and gold_ids.issubset(graph_nodes)
            and bool(topic_ids)
            and bool(graph_edges)
        )

        manifest_rows.append({
            "qid": question_id,
            "split": split,
            "line_number": line_number,
            "number_of_gold_answers": len(gold_ids),
            "number_of_topic_entities": len(topic_ids),
            "number_of_graph_nodes": len(graph_nodes),
            "number_of_graph_edges": len(graph_edges),
            "usable": usable,
        })

        if usable:
            if (
                not PILOT
                or len(selected) < PILOT_QUESTIONS_PER_SPLIT
            ):
                selected.append(
                    (question_id, example)
                )

    selected_examples[split] = selected

    print(
        f"{split}: selected {len(selected)} questions"
    )

    if not PILOT:
        assert len(selected) == expected_full_counts[split], (
            f"Unexpected usable count for {split}: "
            f"{len(selected)} instead of "
            f"{expected_full_counts[split]}"
        )


manifest_path = (
    PROCESSED
    / f"webqsp_manifest_v2_{MODE}.json"
)

manifest_content = {
    "version": "v2_relation_branch",
    "mode": MODE,
    "configuration": {
        "seed": SEED,
        "max_answers": MAX_ANSWERS,
        "max_hops": MAX_HOPS,
        "max_actions_per_trajectory":
            MAX_ACTIONS_PER_TRAJECTORY,
        "max_candidates_per_state":
            MAX_CANDIDATES_PER_STATE,
        "train_trajectories": TRAIN_TRAJECTORIES,
        "eval_trajectories": EVAL_TRAJECTORIES,
    },
    "rows": manifest_rows,
}

manifest_path.write_text(
    json.dumps(
        manifest_content,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8",
)

print("Saved manifest:", manifest_path)


# ============================================================
# 5. GRAPH REPRESENTATION
# ============================================================

def build_grouped_adjacency(edges):
    """
    adjacency[source][relation] = sorted list of targets

    One search action expands every target connected through
    one relation from one source entity.
    """
    adjacency = collections.defaultdict(
        lambda: collections.defaultdict(set)
    )

    for subject, relation, object_entity in edges:
        adjacency[subject][relation].add(object_entity)

    final_adjacency = {}

    for source, relation_map in adjacency.items():
        final_adjacency[source] = {
            relation: sorted(targets)
            for relation, targets in relation_map.items()
        }

    return final_adjacency


def get_frontier_branches(
    visited,
    node_depth,
    expanded_branches,
    adjacency
):
    """
    Return currently available (source, relation, targets)
    branches within the hop limit.
    """
    branches = []

    for source in sorted(visited):
        source_depth = node_depth.get(source, MAX_HOPS + 1)

        if source_depth >= MAX_HOPS:
            continue

        for relation, targets in adjacency.get(
            source,
            {}
        ).items():
            branch_key = (source, relation)

            if branch_key in expanded_branches:
                continue

            branches.append({
                "source": source,
                "relation": relation,
                "targets": targets,
                "source_depth": source_depth,
                "target_depth": source_depth + 1,
            })

    branches.sort(
        key=lambda branch: (
            branch["source_depth"],
            branch["relation"],
            branch["source"],
        )
    )

    return branches


def reachable_unseen_gold(
    initial_targets,
    initial_depth,
    adjacency,
    expanded_branches,
    unseen_gold
):
    """
    Calculate which unseen gold answers could still be reached
    after choosing a candidate branch.

    Gold answers are used only for offline supervision.
    """
    if initial_depth > MAX_HOPS:
        return set()

    reachable_gold = set()
    queue = collections.deque()
    best_depth = {}

    for target in initial_targets:
        previous_depth = best_depth.get(target)

        if previous_depth is None or initial_depth < previous_depth:
            best_depth[target] = initial_depth
            queue.append((target, initial_depth))

    while queue:
        node, depth = queue.popleft()

        if node in unseen_gold:
            reachable_gold.add(node)

        if depth >= MAX_HOPS:
            continue

        for relation, targets in adjacency.get(
            node,
            {}
        ).items():
            if (node, relation) in expanded_branches:
                continue

            next_depth = depth + 1

            for target in targets:
                previous_depth = best_depth.get(target)

                if (
                    previous_depth is None
                    or next_depth < previous_depth
                ):
                    best_depth[target] = next_depth
                    queue.append((target, next_depth))

    return reachable_gold


def tokenize_relation(text):
    return set(
        re.findall(
            r"[a-z0-9]+",
            text.lower().replace("_", " ")
        )
    )


def relation_question_overlap(question, relation_id):
    question_tokens = tokenize_relation(question)
    relation_tokens = tokenize_relation(
        relation_name(relation_id)
    )

    return len(
        question_tokens.intersection(relation_tokens)
    )


# ============================================================
# 6. STATE AND CANDIDATE CREATION
# ============================================================

def make_candidate(
    branch,
    question,
    found_answers,
    visited,
    adjacency,
    expanded_branches,
    history_relations
):
    unseen_gold_placeholder = None

    source = branch["source"]
    relation = branch["relation"]
    targets = branch["targets"]
    source_depth = branch["source_depth"]
    target_depth = branch["target_depth"]

    model_input = (
        f"Question: {question}\n"
        f"Answers found so far: {len(found_answers)}\n"
        f"Visited entities: {len(visited)}\n"
        f"Search step: {len(history_relations)}\n"
        f"Source depth: {source_depth}\n"
        f"Candidate relation: {relation_name(relation)}\n"
        f"Candidate target count: {len(targets)}\n"
        f"Recent relations: "
        f"{' -> '.join(history_relations[-6:]) if history_relations else 'none'}"
    )

    return {
        "source": source,
        "relation": relation,
        "relation_text": relation_name(relation),
        "source_depth": source_depth,
        "target_depth": target_depth,
        "target_count": len(targets),
        "targets": targets,
        "model_input": model_input,
    }


def add_gain_label(
    candidate,
    gold_answers,
    found_answers,
    adjacency,
    expanded_branches
):
    unseen_gold = gold_answers - found_answers

    reachable_answers = reachable_unseen_gold(
        initial_targets=candidate["targets"],
        initial_depth=candidate["target_depth"],
        adjacency=adjacency,
        expanded_branches=expanded_branches,
        unseen_gold=unseen_gold,
    )

    gain = len(reachable_answers)

    if gain >= 3:
        gain_class = "3_plus"
    else:
        gain_class = str(gain)

    candidate["gain"] = gain
    candidate["gain_class"] = gain_class

    return candidate


def select_action(
    candidates,
    policy,
    question,
    rng
):
    if not candidates:
        return None

    if policy == "oracle":
        maximum_gain = max(
            candidate["gain"]
            for candidate in candidates
        )

        best = [
            candidate
            for candidate in candidates
            if candidate["gain"] == maximum_gain
        ]

        best.sort(
            key=lambda candidate: (
                candidate["source_depth"],
                -candidate["target_count"],
                candidate["relation"],
                candidate["source"],
            )
        )

        return best[0]

    if policy == "relation":
        ranked = sorted(
            candidates,
            key=lambda candidate: (
                -relation_question_overlap(
                    question,
                    candidate["relation"]
                ),
                candidate["source_depth"],
                -candidate["target_count"],
                candidate["relation"],
                candidate["source"],
            )
        )

        return ranked[0]

    if policy == "random":
        return candidates[
            rng.randrange(len(candidates))
        ]

    if policy == "fifo":
        return candidates[0]

    raise ValueError(f"Unknown policy: {policy}")


def expand_candidate(
    candidate,
    visited,
    node_depth,
    expanded_branches
):
    source = candidate["source"]
    relation = candidate["relation"]
    target_depth = candidate["target_depth"]

    expanded_branches.add(
        (source, relation)
    )

    newly_visited = set()

    for target in candidate["targets"]:
        if target not in visited:
            visited.add(target)
            newly_visited.add(target)

        old_depth = node_depth.get(target)

        if old_depth is None or target_depth < old_depth:
            node_depth[target] = target_depth

    return newly_visited


def build_trajectory_states(
    question_id,
    example,
    trajectory_id,
    policy
):
    question = get_question(example)
    gold_answers = get_gold_ids(example)
    topic_entities = get_topic_ids(example)

    _, edges = get_graph(example)
    adjacency = build_grouped_adjacency(edges)

    visited = set(topic_entities)

    node_depth = {
        topic_entity: 0
        for topic_entity in topic_entities
    }

    found_answers = visited.intersection(
        gold_answers
    )

    expanded_branches = set()
    history_relations = []
    trajectory_states = []

    rng = random.Random(
        stable_seed(
            f"{question_id}_{trajectory_id}_{policy}"
        )
    )

    for step in range(MAX_ACTIONS_PER_TRAJECTORY):
        branches = get_frontier_branches(
            visited=visited,
            node_depth=node_depth,
            expanded_branches=expanded_branches,
            adjacency=adjacency,
        )

        if not branches:
            break

        candidates = []

        for branch in branches:
            candidate = make_candidate(
                branch=branch,
                question=question,
                found_answers=found_answers,
                visited=visited,
                adjacency=adjacency,
                expanded_branches=expanded_branches,
                history_relations=history_relations,
            )

            candidate = add_gain_label(
                candidate=candidate,
                gold_answers=gold_answers,
                found_answers=found_answers,
                adjacency=adjacency,
                expanded_branches=expanded_branches,
            )

            candidates.append(candidate)

        candidates.sort(
            key=lambda candidate: (
                -candidate["gain"],
                candidate["source_depth"],
                -candidate["target_count"],
                candidate["relation"],
                candidate["source"],
            )
        )

        maximum_gain = max(
            candidate["gain"]
            for candidate in candidates
        )

        stop_label = int(maximum_gain == 0)

        positive_candidates = [
            candidate
            for candidate in candidates
            if candidate["gain"] > 0
        ]

        zero_candidates = [
            candidate
            for candidate in candidates
            if candidate["gain"] == 0
        ]

        rng.shuffle(zero_candidates)

        retained_candidates = (
            positive_candidates
            + zero_candidates
        )[:MAX_CANDIDATES_PER_STATE]

        # At least one positive candidate must remain in CONTINUE states.
        if stop_label == 0:
            assert any(
                candidate["gain"] > 0
                for candidate in retained_candidates
            )

        state = {
            "qid": question_id,
            "question": question,
            "trajectory_id": trajectory_id,
            "trajectory_policy": policy,
            "step": step,
            "found_count": len(found_answers),
            "visited_count": len(visited),
            "expanded_branch_count":
                len(expanded_branches),
            "frontier_count": len(candidates),
            "stop_label": stop_label,
            "candidate_count":
                len(retained_candidates),
            "history_relations":
                history_relations[-8:],
            "candidates": retained_candidates,
        }

        trajectory_states.append(state)

        # A zero-gain state is a valid stopping state.
        if stop_label == 1:
            break

        chosen = select_action(
            candidates=candidates,
            policy=policy,
            question=question,
            rng=rng,
        )

        expand_candidate(
            candidate=chosen,
            visited=visited,
            node_depth=node_depth,
            expanded_branches=expanded_branches,
        )

        history_relations.append(
            chosen["relation_text"]
        )

        found_answers = visited.intersection(
            gold_answers
        )

        # Store a terminal complete-set state when possible.
        if found_answers == gold_answers:
            terminal_branches = get_frontier_branches(
                visited=visited,
                node_depth=node_depth,
                expanded_branches=expanded_branches,
                adjacency=adjacency,
            )

            if terminal_branches:
                terminal_candidates = []

                for branch in terminal_branches:
                    candidate = make_candidate(
                        branch=branch,
                        question=question,
                        found_answers=found_answers,
                        visited=visited,
                        adjacency=adjacency,
                        expanded_branches=expanded_branches,
                        history_relations=history_relations,
                    )

                    candidate = add_gain_label(
                        candidate=candidate,
                        gold_answers=gold_answers,
                        found_answers=found_answers,
                        adjacency=adjacency,
                        expanded_branches=expanded_branches,
                    )

                    terminal_candidates.append(candidate)

                terminal_candidates.sort(
                    key=lambda candidate: (
                        candidate["source_depth"],
                        candidate["relation"],
                        candidate["source"],
                    )
                )

                terminal_candidates = terminal_candidates[
                    :MAX_CANDIDATES_PER_STATE
                ]

                trajectory_states.append({
                    "qid": question_id,
                    "question": question,
                    "trajectory_id": trajectory_id,
                    "trajectory_policy": policy,
                    "step": step + 1,
                    "found_count": len(found_answers),
                    "visited_count": len(visited),
                    "expanded_branch_count":
                        len(expanded_branches),
                    "frontier_count":
                        len(terminal_branches),
                    "stop_label": 1,
                    "candidate_count":
                        len(terminal_candidates),
                    "history_relations":
                        history_relations[-8:],
                    "candidates":
                        terminal_candidates,
                })

            break

    return trajectory_states


# ============================================================
# 7. GENERATE STATES
# ============================================================

training_policies = [
    "oracle",
    "relation",
    "random",
]

evaluation_policies = [
    "oracle",
]

state_paths = {}
generation_summary = {}

generation_start = time.time()

for split, examples in selected_examples.items():
    output_path = (
        PROCESSED
        / f"webqsp_{split}_states_v2_{MODE}.jsonl"
    )

    state_paths[split] = output_path

    number_of_states = 0
    number_of_candidates = 0
    number_of_positive_candidates = 0
    number_of_stop_states = 0
    number_of_continue_states = 0

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as output_file:

        for example_number, (question_id, example) in enumerate(
            examples,
            start=1
        ):
            if split == "train":
                policies = training_policies[
                    :TRAIN_TRAJECTORIES
                ]
            else:
                policies = evaluation_policies[
                    :EVAL_TRAJECTORIES
                ]

            for trajectory_id, policy in enumerate(policies):
                states = build_trajectory_states(
                    question_id=question_id,
                    example=example,
                    trajectory_id=trajectory_id,
                    policy=policy,
                )

                for state in states:
                    output_file.write(
                        json.dumps(
                            state,
                            ensure_ascii=False
                        )
                        + "\n"
                    )

                    number_of_states += 1
                    number_of_candidates += (
                        state["candidate_count"]
                    )

                    number_of_positive_candidates += sum(
                        candidate["gain"] > 0
                        for candidate
                        in state["candidates"]
                    )

                    if state["stop_label"] == 1:
                        number_of_stop_states += 1
                    else:
                        number_of_continue_states += 1

            if example_number % 100 == 0:
                print(
                    f"{split}: processed "
                    f"{example_number}/{len(examples)}"
                )

    generation_summary[split] = {
        "questions": len(examples),
        "states": number_of_states,
        "candidates": number_of_candidates,
        "positive_candidates":
            number_of_positive_candidates,
        "stop_states": number_of_stop_states,
        "continue_states": number_of_continue_states,
        "output_path": str(output_path),
    }

    print(
        f"\n{split}:",
        generation_summary[split]
    )


generation_summary_path = (
    OUTPUTS
    / f"state_generation_summary_v2_{MODE}.json"
)

generation_summary_path.write_text(
    json.dumps(
        generation_summary,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8",
)

print(
    "\nState-generation time:",
    round((time.time() - generation_start) / 60, 2),
    "minutes"
)


# ============================================================
# 8. INTEGRITY AND LEAKAGE AUDIT
# ============================================================

forbidden_patterns = [
    r"(?im)^\s*gold(?:\s+answers?|\s+entities?|\s+ids?)?\s*:",
    r"(?im)^\s*gain\s*:",
    r"(?im)^\s*gain_class\s*:",
    r"(?im)^\s*stop_label\s*:",
    r"(?im)^\s*target_label\s*:",
    r"(?im)^\s*oracle_gain\s*:",
]

integrity_audit = {}

for split, path in state_paths.items():
    gain_classes = collections.Counter()
    stop_labels = collections.Counter()
    policies = collections.Counter()

    number_of_states = 0
    number_of_candidates = 0
    leakage_examples = []

    for state in iter_jsonl(path):
        number_of_states += 1

        stop_labels[
            str(state["stop_label"])
        ] += 1

        policies[
            state["trajectory_policy"]
        ] += 1

        candidate_gains = [
            candidate["gain"]
            for candidate in state["candidates"]
        ]

        expected_stop = int(
            max(candidate_gains, default=0) == 0
        )

        assert state["stop_label"] == expected_stop, (
            f"Inconsistent stop label: "
            f"{state['qid']}"
        )

        for candidate in state["candidates"]:
            number_of_candidates += 1
            gain_classes[
                candidate["gain_class"]
            ] += 1

            model_input = candidate.get(
                "model_input",
                ""
            )

            matched = [
                pattern
                for pattern in forbidden_patterns
                if re.search(pattern, model_input)
            ]

            if matched:
                leakage_examples.append({
                    "qid": state["qid"],
                    "patterns": matched,
                    "model_input": model_input,
                })

    assert number_of_states > 0, (
        f"No states found in {path}"
    )

    integrity_audit[split] = {
        "states": number_of_states,
        "candidates": number_of_candidates,
        "gain_classes": dict(gain_classes),
        "stop_labels": dict(stop_labels),
        "trajectory_policies": dict(policies),
        "leakage_flags":
            len(leakage_examples),
        "leakage_examples":
            leakage_examples[:5],
    }


assert all(
    result["leakage_flags"] == 0
    for result in integrity_audit.values()
), "Explicit target leakage was detected"

integrity_path = (
    OUTPUTS
    / f"state_integrity_audit_v2_{MODE}.json"
)

integrity_path.write_text(
    json.dumps(
        integrity_audit,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8",
)

print("\nIntegrity audit")
print(json.dumps(
    integrity_audit,
    indent=2,
    ensure_ascii=False
))
print("Integrity audit passed.")


# ============================================================
# 9. SYMBOLIC BASELINES
# ============================================================

def simulate_search(
    example,
    policy,
    action_budget,
    patience=None,
    simulation_seed=42
):
    question = get_question(example)
    gold_answers = get_gold_ids(example)
    topic_entities = get_topic_ids(example)

    _, edges = get_graph(example)
    adjacency = build_grouped_adjacency(edges)

    visited = set(topic_entities)

    node_depth = {
        topic_entity: 0
        for topic_entity in topic_entities
    }

    found_answers = visited.intersection(
        gold_answers
    )

    expanded_branches = set()
    history_relations = []

    rng = random.Random(simulation_seed)

    number_of_actions = 0
    number_of_retrieved_edges = 0
    consecutive_zero_gain_actions = 0

    while number_of_actions < action_budget:
        branches = get_frontier_branches(
            visited=visited,
            node_depth=node_depth,
            expanded_branches=expanded_branches,
            adjacency=adjacency,
        )

        if not branches:
            break

        candidates = []

        for branch in branches:
            candidate = make_candidate(
                branch=branch,
                question=question,
                found_answers=found_answers,
                visited=visited,
                adjacency=adjacency,
                expanded_branches=expanded_branches,
                history_relations=history_relations,
            )

            candidate = add_gain_label(
                candidate=candidate,
                gold_answers=gold_answers,
                found_answers=found_answers,
                adjacency=adjacency,
                expanded_branches=expanded_branches,
            )

            candidates.append(candidate)

        candidates.sort(
            key=lambda candidate: (
                candidate["source_depth"],
                candidate["relation"],
                candidate["source"],
            )
        )

        chosen = select_action(
            candidates=candidates,
            policy=policy,
            question=question,
            rng=rng,
        )

        if chosen is None:
            break

        # Oracle stops when no remaining branch can add an answer.
        if policy == "oracle" and chosen["gain"] == 0:
            break

        previous_found_count = len(found_answers)

        expand_candidate(
            candidate=chosen,
            visited=visited,
            node_depth=node_depth,
            expanded_branches=expanded_branches,
        )

        history_relations.append(
            chosen["relation_text"]
        )

        number_of_actions += 1
        number_of_retrieved_edges += (
            chosen["target_count"]
        )

        found_answers = visited.intersection(
            gold_answers
        )

        if len(found_answers) == previous_found_count:
            consecutive_zero_gain_actions += 1
        else:
            consecutive_zero_gain_actions = 0

        if found_answers == gold_answers:
            break

        if (
            patience is not None
            and consecutive_zero_gain_actions >= patience
        ):
            break

    recall = (
        len(found_answers) / len(gold_answers)
        if gold_answers
        else 0.0
    )

    # In the oracle answer-recognition experiment, every
    # returned answer is a gold answer, so precision is 1
    # whenever at least one answer is found.
    precision = (
        1.0 if found_answers else 0.0
    )

    if precision + recall == 0:
        answer_f1 = 0.0
    else:
        answer_f1 = (
            2 * precision * recall
            / (precision + recall)
        )

    return {
        "f1": answer_f1,
        "recall": recall,
        "exact": int(
            found_answers == gold_answers
        ),
        "actions": number_of_actions,
        "retrieved_edges":
            number_of_retrieved_edges,
        "premature": int(
            found_answers != gold_answers
        ),
    }


baseline_configurations = [
    {
        "name": "fifo_b8",
        "policy": "fifo",
        "budget": 8,
        "patience": None,
    },
    {
        "name": "fifo_b16",
        "policy": "fifo",
        "budget": 16,
        "patience": None,
    },
    {
        "name": "fifo_b24",
        "policy": "fifo",
        "budget": 24,
        "patience": None,
    },
    {
        "name": "patience1",
        "policy": "fifo",
        "budget": 24,
        "patience": 1,
    },
    {
        "name": "patience2",
        "policy": "fifo",
        "budget": 24,
        "patience": 2,
    },
    {
        "name": "relation_b16",
        "policy": "relation",
        "budget": 16,
        "patience": None,
    },
    {
        "name": "relation_b24",
        "policy": "relation",
        "budget": 24,
        "patience": None,
    },
    {
        "name": "random_b16",
        "policy": "random",
        "budget": 16,
        "patience": None,
    },
    {
        "name": "oracle_gain_b16",
        "policy": "oracle",
        "budget": 16,
        "patience": None,
    },
    {
        "name": "oracle_gain_b24",
        "policy": "oracle",
        "budget": 24,
        "patience": None,
    },
]


baseline_results = {}

for split in ["dev", "test"]:
    examples = selected_examples[split]
    baseline_results[split] = {}

    for configuration in baseline_configurations:
        measurements = []

        for question_number, (
            question_id,
            example
        ) in enumerate(examples):
            measurement = simulate_search(
                example=example,
                policy=configuration["policy"],
                action_budget=configuration["budget"],
                patience=configuration["patience"],
                simulation_seed=stable_seed(
                    f"{split}_{question_id}_"
                    f"{configuration['name']}"
                ),
            )

            measurements.append(measurement)

        metrics = {}

        for metric_name in measurements[0]:
            metric_values = [
                measurement[metric_name]
                for measurement in measurements
            ]

            metrics[metric_name] = float(
                np.mean(metric_values)
            )

        baseline_results[split][
            configuration["name"]
        ] = metrics


baseline_path = (
    OUTPUTS
    / f"symbolic_baselines_v2_{MODE}.json"
)

baseline_payload = {
    "version": "v2_relation_branch",
    "mode": MODE,
    "action_definition":
        "(source entity, relation) branch",
    "configurations":
        baseline_configurations,
    "results":
        baseline_results,
}

baseline_path.write_text(
    json.dumps(
        baseline_payload,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8",
)

print("\nSymbolic baseline results")
print(json.dumps(
    baseline_results,
    indent=2,
    ensure_ascii=False
))


# ============================================================
# 10. ARTIFACT INVENTORY
# ============================================================

artifact_paths = [
    manifest_path,
    *state_paths.values(),
    generation_summary_path,
    integrity_path,
    baseline_path,
]

inventory = []

print("\nGenerated artifacts")
print("-" * 90)

for artifact_path in artifact_paths:
    file_hash = hashlib.sha256(
        artifact_path.read_bytes()
    ).hexdigest()

    file_size = artifact_path.stat().st_size

    inventory.append({
        "path": str(artifact_path),
        "bytes": file_size,
        "megabytes": round(
            file_size / (1024 ** 2),
            4
        ),
        "sha256": file_hash,
    })

    print(
        artifact_path.name,
        f"{file_size / (1024 ** 2):.2f} MB",
        file_hash[:16],
    )


inventory_path = (
    OUTPUTS
    / f"artifact_inventory_v2_{MODE}.json"
)

inventory_path.write_text(
    json.dumps(
        inventory,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8",
)


# ============================================================
# 11. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 90)
print("SETSTOP-KGQA V2 PREPROCESSING COMPLETED")
print("=" * 90)
print("Mode:", MODE.upper())
print("Manifest:", manifest_path)
print("Generation summary:", generation_summary_path)
print("Integrity audit:", integrity_path)
print("Baselines:", baseline_path)
print("Inventory:", inventory_path)

if PILOT:
    print("\nPilot completed.")
    print("Send the V2 pilot output for verification.")
    print("Do not start the full run until the oracle results are checked.")
else:
    print("\nFull preprocessing completed.")
    print("The next stage is controller training and dev-threshold calibration.")

print("=" * 90)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SETSTOP-KGQA PREPROCESSING V2
Runtime: CPU
Mode: FULL
Data: /content/drive/MyDrive/SetStop_KGQA/data/extracted/webqsp
Maximum graph hops: 3
Action definition: (source entity, relation) branch

Schema check
------------------------------------------------------------------------------------------
Top-level keys: ['answers', 'entities', 'id', 'question', 'subgraph']
Question: what is the name of justin bieber brother
Topic entities: 1
Gold answers: 1
Graph entities: 2000
Graph edges: 9088
Schema check passed.
train: selected 1200 questions
dev: selected 118 questions
test: selected 736 questions
Saved manifest: /content/drive/MyDrive/SetStop_KGQA/data/processed/webqsp_manifest_v2_full.json
train: processed 100/1200
train: processed 200/1200
train: processed 300/1200
train: processed 400/1200
train: processed 500/1200
train: processed 600/1200
train: processed 7